#### Librerias

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import re
import random
import joblib
import nltk
from nltk.corpus import stopwords

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Variables

In [2]:
MODELOS_PATH = "/content/drive/MyDrive/Trabajo práctico 3/modelos"

#### Carga de modelo y vectorizador (LR optimizado)

In [3]:
modelo = joblib.load(f"{MODELOS_PATH}/modelo_lr_optimizado.pkl")
vectorizer = joblib.load(f"{MODELOS_PATH}/vectorizer_tfidf.pkl")

#### Función de limpieza

Se repite acá porque cada notebook de Colab es independiente.

In [4]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def limpieza(texto):
    texto = texto.lower()
    texto = re.sub(r'http\S+|www\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'#', '', texto)
    texto = re.sub(r'[^a-z\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    tokens = [t for t in texto.split() if t not in stop_words]
    return ' '.join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### Preguntas de la cita

In [10]:
preguntas = [
    "¿Cómo me ves hoy? ¿Estoy linda?",
    "¿Cuál es lo más vergonzoso que te pasó en una cita?",
    "¿Alguna vez te hicieron ghosting? ¿Cómo te sentiste?",
    "¿Qué diría tu ex sobre vos?",
    "Describí tu peor primera cita de la historia.",
    "¿Qué hábito tuyo podría molestarle a una pareja?",
    "¿Cómo te sentís cuando te preguntan 'a dónde va esto'?",
    "¿Cuál es tu mayor miedo con las citas en este momento?",
]

#### Clasificación con umbral de confianza

In [6]:
!pip install deep-translator -q
from deep_translator import GoogleTranslator

def clasificar_respuesta(texto, umbral=0.60):
    texto_en = GoogleTranslator(source='es', target='en').translate(texto)
    texto_limpio = limpieza(texto_en)
    vector = vectorizer.transform([texto_limpio])
    proba = modelo.predict_proba(vector)[0]

    if max(proba) < umbral:
        return "Dudoso 😐", proba, texto_en
    elif proba[1] > proba[0]:
        return "Positivo 😊", proba, texto_en
    else:
        return "Negativo 😡", proba, texto_en

#### El juego

In [7]:
def jugar_cita(n_rondas=5):
    print("Bienvenido/a a 'Speed Dating con IA' 💘")
    print("Te voy a hacer algunas preguntas incómodas. Contestá con sinceridad...\n")
    preguntas_ronda = random.sample(preguntas, n_rondas)
    for pregunta in preguntas_ronda:
        print(f"\n💬 {pregunta}")
        respuesta = input("Tu respuesta: ")
        resultado, proba, texto_en = clasificar_respuesta(respuesta)
        print(f"El modelo lee tu vibra como: {resultado} (confianza: {max(proba)*100:.1f}%)")
    print("\n¡Gracias por la cita! 😉")

#### Correr el juego

In [11]:
jugar_cita()

Bienvenido/a a 'Speed Dating con IA' 💘
Te voy a hacer algunas preguntas incómodas. Contestá con sinceridad...


💬 ¿Cómo te sentís cuando te preguntan 'a dónde va esto'?
Tu respuesta: No se la verdad
El modelo lee tu vibra como: Dudoso 😐 (confianza: 52.1%)

💬 ¿Alguna vez te hicieron ghosting? ¿Cómo te sentiste?
Tu respuesta: si muchas veces, me dio igual
El modelo lee tu vibra como: Dudoso 😐 (confianza: 51.2%)

💬 ¿Cuál es tu mayor miedo con las citas en este momento?
Tu respuesta: Que me corran la cara cuando le lance un beso
El modelo lee tu vibra como: Positivo 😊 (confianza: 63.5%)

💬 Describí tu peor primera cita de la historia.
Tu respuesta: Salio corriendo apenas me vio
El modelo lee tu vibra como: Negativo 😡 (confianza: 81.2%)

💬 ¿Cómo me ves hoy? ¿Estoy linda?
Tu respuesta: No, estas fea
El modelo lee tu vibra como: Negativo 😡 (confianza: 68.0%)

¡Gracias por la cita! 😉
